In [1]:
import glob
import numpy as np
from multiprocessing import Pool
import os

In [2]:
# Path to your 21,638 chips
data_path = '/explore/nobackup/projects/pix4dcloud/aliewehr/chipTests/chips/allChips'
files = glob.glob(data_path + '/*/*.npz')
print(f'Found {len(files)} chips.')

# Calculate transect length stats directly from filenames (Super fast!)
lengths = []
for f in files:
    basename = os.path.basename(f)
    if '_len' in basename:
        try:
            # Extract the number right after '_len' and before '.npz'
            length_str = basename.split('_len')[1].split('.npz')[0]
            lengths.append(int(length_str))
        except:
            pass

if lengths:
    print(f'\n--- TRANSECT LENGTH STATS ---')
    print(f'Minimum length: {min(lengths)}')
    print(f'Maximum length: {max(lengths)}')
    print(f'Average length: {sum(lengths)/len(lengths):.1f}')

Found 21638 chips.

--- TRANSECT LENGTH STATS ---
Minimum length: 478
Maximum length: 621
Average length: 522.1


In [3]:
def get_min_max(file_path):
    try:
        # Load the file
        chip = np.load(file_path, allow_pickle=True)
        
        # Check for the correct key
        key = 'ABI/chip' if 'ABI/chip' in chip else 'chip'
        data = chip[key]
        
        # Calculate min and max for each of the 16 channels across Time, Height, Width (axes 0, 1, 2)
        # Use np.nanmax and np.nanmin to ignore any missing NaN values in the dataset
        c_max = np.nanmax(data, axis=(0, 1, 2))
        c_min = np.nanmin(data, axis=(0, 1, 2))
        
        return c_min, c_max
    except Exception as e:
        print(f'Error reading {file_path}: {e}')
        return None

In [4]:
global_mins = np.full(16, np.inf)
global_maxs = np.full(16, -np.inf)

print('\nStarting min/max processing...')
# Use multiprocessing to read all 21k files much faster. 
# 16 workers is usually a good number for the ADAPT cluster nodes.
with Pool(16) as p:
    # We loop through using enumerate so we can print progress manually
    for i, res in enumerate(p.imap_unordered(get_min_max, files)):
        if res is not None:
            c_min, c_max = res
            # np.fmin and np.fmax safely ignore NaNs if an entire file happened to be NaN
            global_mins = np.fmin(global_mins, c_min)
            global_maxs = np.fmax(global_maxs, c_max)
            
        # Simple progress tracker instead of tqdm
        if (i + 1) % 1000 == 0 or (i + 1) == len(files):
            print(f'Processed {i + 1} / {len(files)} chips...')

print('Done processing!')


Starting min/max processing...
Processed 1000 / 21638 chips...
Processed 2000 / 21638 chips...
Processed 3000 / 21638 chips...
Processed 4000 / 21638 chips...
Processed 5000 / 21638 chips...
Processed 6000 / 21638 chips...
Processed 7000 / 21638 chips...
Processed 8000 / 21638 chips...
Processed 9000 / 21638 chips...
Processed 10000 / 21638 chips...
Processed 11000 / 21638 chips...
Processed 12000 / 21638 chips...
Processed 13000 / 21638 chips...
Processed 14000 / 21638 chips...
Processed 15000 / 21638 chips...
Processed 16000 / 21638 chips...
Processed 17000 / 21638 chips...
Processed 18000 / 21638 chips...
Processed 19000 / 21638 chips...
Processed 20000 / 21638 chips...
Processed 21000 / 21638 chips...
Processed 21638 / 21638 chips...
Done processing!


In [5]:
print('\n--- GLOBAL MINIMUMS ---')
print(list(global_mins))
print('\n--- GLOBAL MAXIMUMS ---')
print(list(global_maxs))

print('\n\nCopy the following directly into transforms.py:')
print(f'self.min_vals = np.array({list(global_mins)}, dtype=np.float32)')
print(f'self.max_vals = np.array({list(global_maxs)}, dtype=np.float32)')


--- GLOBAL MINIMUMS ---
[-25.936647415161133, -20.2899112701416, -12.037643432617188, -4.522368431091309, -3.0596137046813965, -0.9609506726264954, -0.03759999945759773, 0.1447715163230896, -0.8235999941825867, -0.9560999870300293, -1.3021999597549438, -1.5393999814987183, -1.6442999839782715, 5.90310001373291, -1.7558000087738037, -5.239200115203857]

--- GLOBAL MAXIMUMS ---
[804.0360717773438, 628.9872436523438, 373.1669921875, 140.19342041015625, 94.84803009033203, 29.789470672607422, 25.589599609375, 9.452010154724121, 45.29140090942383, 81.0928955078125, 135.2646026611328, 109.84480285644531, 185.56988525390625, 200.9023895263672, 214.30140686035156, 174.69261169433594]


Copy the following directly into transforms.py:
self.min_vals = np.array([-25.936647415161133, -20.2899112701416, -12.037643432617188, -4.522368431091309, -3.0596137046813965, -0.9609506726264954, -0.03759999945759773, 0.1447715163230896, -0.8235999941825867, -0.9560999870300293, -1.3021999597549438, -1.53939998